In [1]:
import sys
from pathlib import Path

from market_inventory.universe import CoinUniverse, ProjectUniverse
from market_inventory.inventory import inventory_crypto_markets
from market_inventory.polymarket_clients import GammaClient


coin_universe = CoinUniverse.from_json()
project_universe = ProjectUniverse.from_json()
gamma = GammaClient()

df = inventory_crypto_markets(
    gamma=gamma,
    coin_universe=coin_universe,
    project_universe=project_universe,
    limit_events=500,
)

In [8]:
df[df["resolution_source"].isna()| df["resolution_source"].str.contains("http")]

,market,kind,symbol,metric,resolution_date,resolution_source,resolution_terms,resolution_data_type,resolution_interval,interval_source,routing_notes
0,Will El Salvador hold $1b+ of BTC by September 30?,edge,BTC,price,2025-09-30 00:00:00+00:00,https://intel.arkm.com/explorer/entity/el-salvador,"This market will resolve to “Yes” if the bitcoin holdings officially owned by the Republic of the El Salvador government reaches or surpasses a value of $1b at any point by September 30, 2025, 11:59 PM ET. Otherwise, this market will resolve to “No”.\n\nThe primary resolutions source for this market will be the ARKHAM INTEL tracker (see: https://intel.arkm.com/explorer/entity/el-salvador). Any temporary glitches or errors in the tracker will not be considered. If the tracker becomes permanently unavailable another credible source may be used. Official announcements form the government of El Salvador confirming their bitcoin holdings have reached or surpassed $1b in value will also qualify.",other,NaN,none,no clear candle/daily/ranking/aggregate/equity signals
1,Will El Salvador hold $1b+ of BTC by December 31?,edge,BTC,price,2025-09-30 00:00:00+00:00,https://intel.arkm.com/explorer/entity/el-salvador,"This market will resolve to “Yes” if the bitcoin holdings officially owned by the Republic of the El Salvador government reaches or surpasses a value of $1b at any point by December 31, 2025, 11:59 PM ET. Otherwise, this market will resolve to “No”.\n\nThe primary resolutions source for this market will be the ARKHAM INTEL tracker (see: https://intel.arkm.com/explorer/entity/el-salvador). Any temporary glitches or errors in the tracker will not be considered. If the tracker becomes permanently unavailable another credible source may be used. Official announcements form the government of El Salvador confirming their bitcoin holdings have reached or surpassed $1b in value will also qualify.",other,NaN,none,no clear candle/daily/ranking/aggregate/equity signals
3,MegaETH market cap (FDV) >$2B one day after launch?,edge,MegaETH,fdv,2025-12-31 00:00:00+00:00,NaN,"This market will resolve to ""Yes"" if the Fully Diluted Valuation of MegaETH's token is greater than $2,000,000,000 1 day after launch. Otherwise, the market will resolve to ""No.""\n\nFor the purposes of this market ""locked"" tokens or non-swappable tokens will not be considered a launch.\n\n""1 day after launch"" is defined as 24 hours after launch. The resolution source for this market is the most liquid price source available. If MegaETH doesn't launch a token by June 30, 2026, 11:59 PM ET, this market will resolve to ""No.""",daily_metric,1d,explicit,daily metric / historical data resolution
4,MegaETH market cap (FDV) >$1B one day after launch?,edge,MegaETH,fdv,2025-12-31 00:00:00+00:00,NaN,"This market will resolve to ""Yes"" if the Fully Diluted Valuation of MegaETH's token is greater than $1,000,000,000 1 day after launch. Otherwise, the market will resolve to ""No.""\n\nFor the purposes of this market ""locked"" tokens or non-swappable tokens will not be considered a launch.\n\n""1 day after launch"" is defined as 24 hours after launch. The resolution source for this market is the most liquid price source available. If MegaETH doesn't launch a token by June 30, 2026, 11:59 PM ET, this market will resolve to ""No.""",daily_metric,1d,explicit,daily metric / historical data resolution
5,MegaETH market cap (FDV) >$1.5B one day after launch?,edge,MegaETH,fdv,2025-12-31 00:00:00+00:00,NaN,"This market will resolve to ""Yes"" if the Fully Diluted Valuation of MegaETH's token is greater than $1,500,000,000 1 day after launch. Otherwise, the market will resolve to ""No.""\n\nFor the purposes of this market ""locked"" tokens or non-swappable tokens will not be considered a launch.\n\n""1 day after launch"" is defined as 24 hours after launch. The resolution source for this market is the most liquid price source available. If MegaETH doesn't launch a token by June 30, 2026, 11:59 PM ET,

In [7]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

In [3]:
import logging

logging.basicConfig(
    level=logging.INFO,   # set DEBUG for even more detail
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)

In [ ]:
from pipeline.historical_triage_runner import triage_dataframe_async

df_out = await triage_dataframe_async(
    df,
    max_rows=1,
    max_concurrency=1,
    row_timeout_s=60.0,
    total_timeout_s=300.0,
    log_every=1,
)

2026-02-08 12:08:59,704 | INFO | pipeline.historical_triage_runner | triage_dataframe_async start: rows=500 max_concurrency=1 only_source_nan_or_url=True max_rows=1
2026-02-08 12:08:59,710 | INFO | pipeline.historical_triage_runner | after filter: rows=241
2026-02-08 12:08:59,712 | INFO | pipeline.historical_triage_runner | after max_rows=1 cap: rows=1
2026-02-08 12:09:05,379 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"
2026-02-08 12:09:56,039 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200 OK"
2026-02-08 12:09:56,214 | INFO | pm_agents.historical_data_triage_agent | triage_market_row ok: market_id= relevance=yes feasibility=maybe paywall=possible (56.50s)
2026-02-08 12:09:56,214 | INFO | pipeline.historical_triage_runner | progress: 1/1 completed (1 ok, 0 errors)
2026-02-08 12:09:56,215 | INFO | pipeline.historical_triage_runner | triage completed: ok=1 errors=0
2026-02-08 12:09:56,217 | INF

2026-02-08 12:10:00,752 | INFO | httpx | HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1 204 No Content"


In [10]:
df_out.to_csv("triage_output.csv", index=False)